<a href="https://colab.research.google.com/github/DaniNar2/Aspect-Based-Sentiment-Analysis/blob/main/ATE_Restaurant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements

In [ ]:
!pip install transformers datasets seqeval nltk -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

--2026-06-28 07:22:31--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-06-28 07:22:31--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-06-28 07:22:31--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

# Importing libraries

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.tokenize import TreebankWordTokenizer
nltk.download('punkt')
from collections import Counter
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification, pipeline
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from tqdm import tqdm
from pathlib import Path

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


# Setup

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

# Functions

In [ ]:
# Creating BIO labels:

def create_bio_labels(sentence, aspects):
    spans = list(tokenizer.span_tokenize(sentence))
    tokens = [
        sentence[start:end]
        for start, end in spans
    ]
    labels = ["O"] * len(tokens)
    for aspect, aspect_start, aspect_end in aspects:
        aspect_tokens = []
        for i, (token_start, token_end) in enumerate(spans):
            overlap = (
                token_start < aspect_end
                and token_end > aspect_start
            )
            if overlap:
                aspect_tokens.append(i)
        if len(aspect_tokens) > 0:
            labels[aspect_tokens[0]] = "B-ASP"
            for idx in aspect_tokens[1:]:
                labels[idx] = "I-ASP"
    return tokens, labels

In [ ]:
# Counting aspects:

def count_aspects(data):
    count = 0
    for sample in data:
        count += sum(
            1
            for label in sample["labels"]
            if label == "B-ASP"
        )
    return count

# Dataset analysis

In [ ]:
drive.mount('/content/drive')
DATA = "/content/drive/MyDrive/SII/Restaurants_Train_v2.csv"
df = pd.read_csv(DATA)
MODEL_DIR = Path(DATA).parent

Mounted at /content/drive


In [ ]:
print(df.columns)

Index(['id', 'Sentence', 'Aspect Term', 'polarity', 'from', 'to'], dtype='object')


In [ ]:
print(df.shape)

(3693, 6)


In [ ]:
df.head()

,id,Sentence,Aspect Term,polarity,from,to
0,3121,But the staff was so horrible to us.,staff,negative,8,13
1,2777,"To be completely fair, the only redeeming fact...",food,positive,57,61
2,1634,"The food is uniformly exceptional, with a very...",food,positive,4,8
3,1634,"The food is uniformly exceptional, with a very...",kitchen,positive,55,62
4,1634,"The food is uniformly exceptional, with a very...",menu,neutral,141,145


# Data pre-processing

In [ ]:
grouped = []

for sentence, group in df.groupby("Sentence"):
    aspects = []
    for _, row in group.iterrows():
        aspects.append(
            (
                row["Aspect Term"],
                int(row["from"]),
                int(row["to"])
            )
        )
    grouped.append({
        "sentence": sentence,
        "aspects": aspects
    })

print(len(grouped))

2019


In [ ]:
tokenizer = TreebankWordTokenizer()

In [ ]:
example_sentence = grouped[0]["sentence"]
spans = list(tokenizer.span_tokenize(example_sentence))
tokens = [example_sentence[start:end] for start, end in spans]
for token, span in zip(tokens, spans):
    print(token, span)

$ (0, 1)
160 (1, 4)
for (5, 8)
2 (9, 10)
filets (11, 17)
, (17, 18)
2 (19, 20)
sides (21, 26)
, (26, 27)
an (28, 30)
appetizer (31, 40)
and (41, 44)
drinks (45, 51)
. (51, 52)


In [ ]:
processed = []

for item in grouped:
    sentence = item["sentence"]
    tokens, labels = create_bio_labels(
        sentence,
        item["aspects"]
    )
    processed.append({
        "sentence": sentence,
        "tokens": tokens,
        "labels": labels
    })

In [ ]:
sample = processed[0]
for t, l in zip(sample["tokens"], sample["labels"]):
    print(f"{t:15} {l}")

$               O
160             O
for             O
2               O
filets          B-ASP
,               O
2               O
sides           B-ASP
,               O
an              O
appetizer       B-ASP
and             O
drinks          B-ASP
.               O


In [ ]:
train_data, temp_data = train_test_split(
    processed,
    test_size=0.2,
    random_state=42
)

val_data, test_data = train_test_split(
    temp_data,
    test_size=0.5,
    random_state=42
)

In [ ]:
counter = Counter()
for sample in train_data:
    counter.update(sample["labels"])
print(counter)

Counter({'O': 23866, 'B-ASP': 2975, 'I-ASP': 1163})


In [ ]:
print(len(train_data))
print(len(val_data))
print(len(test_data))

1615
202
202


In [ ]:
print("Train:", count_aspects(train_data))
print("Val:", count_aspects(val_data))
print("Test:", count_aspects(test_data))

Train: 2975
Val: 330
Test: 385


# Dictionaries

In [ ]:
word2idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

In [ ]:
for sample in train_data:
    for token in sample["tokens"]:
        if token not in word2idx:
            word2idx[token] = len(word2idx)

In [ ]:
label2idx = {
    "O": 0,
    "B-ASP": 1,
    "I-ASP": 2,
    "PAD": -100
}

idx2label = {
    v:k
    for k,v in label2idx.items()
}

In [ ]:
idx2label = {
    0: "O",
    1: "B-ASP",
    2: "I-ASP"
}

# Pre-trained embedding

In [ ]:
embedding_dim = 300
embeddings_index = {}

with open(
    "glove.6B.300d.txt",
    encoding="utf8"
) as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(
            values[1:],
            dtype="float32"
        )
        embeddings_index[word] = vector

print(len(embeddings_index))

400000


In [ ]:
embedding_matrix = np.random.normal(
    scale=0.6,
    size=(len(word2idx), embedding_dim)
)

for word, idx in word2idx.items():
    vector = embeddings_index.get(word.lower())
    if vector is not None:
        embedding_matrix[idx] = vector

embedding_matrix = torch.tensor(
    embedding_matrix,
    dtype=torch.float
)

print(embedding_matrix.shape)

torch.Size([3940, 300])


# Dataset creation

In [ ]:
class AspectDataset(Dataset):

    def __init__(
        self,
        data,
        word2idx,
        label2idx,
        max_len=100
    ):

        self.data = data
        self.word2idx = word2idx
        self.label2idx = label2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        sample = self.data[idx]

        tokens = sample["tokens"]
        labels = sample["labels"]

        token_ids = [
            self.word2idx.get(token, self.word2idx["<UNK>"])
            for token in tokens
        ]

        label_ids = [
            self.label2idx[label]
            for label in labels
        ]

        attention_mask = [1] * len(token_ids)

        if len(token_ids) < self.max_len:

            padding = self.max_len - len(token_ids)

            token_ids += [0] * padding
            label_ids += [-100] * padding
            attention_mask += [0] * padding

        else:

            token_ids = token_ids[:self.max_len]
            label_ids = label_ids[:self.max_len]
            attention_mask = attention_mask[:self.max_len]

        return {
            "input_ids": torch.tensor(token_ids),
            "labels": torch.tensor(label_ids),
            "mask": torch.tensor(attention_mask)
        }

In [ ]:
# Train dataset

train_dataset = AspectDataset(
    train_data,
    word2idx,
    label2idx
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [ ]:
# Test dataset

test_dataset = AspectDataset(
    test_data,
    word2idx,
    label2idx
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32
)

In [ ]:
# Validation dataset

val_dataset = AspectDataset(
    val_data,
    word2idx,
    label2idx
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32
)

# Models

In [ ]:
# RNN with random embedding

class RNN_1(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=100,
        hidden_dim=128,
        num_labels=3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            num_labels
        )

    def forward(self, x):

        x = self.embedding(x)

        out, _ = self.rnn(x)

        logits = self.fc(out)

        return logits

In [ ]:
# RNN with pre-trained embedding

class RNN_2(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_matrix=None,
        embedding_dim=300,
        hidden_dim=256,
        num_labels=3
    ):

        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
                embedding_matrix,
                freeze=False,
                padding_idx=0
            )

        embedding_dim = embedding_matrix.shape[1]

        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(
            hidden_dim,
            num_labels
        )

    def forward(self, x):

        x = self.embedding(x)

        x, _ = self.rnn(x)

        x = self.dropout(x)

        x = self.fc(x)

        return x

In [ ]:
# LSTM with random embedding

class LSTM_1(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=100,
        hidden_dim=128,
        num_labels=3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            num_labels
        )

    def forward(self, x):

        x = self.embedding(x)

        out, _ = self.lstm(x)

        logits = self.fc(out)

        return logits

In [ ]:
# LSTM with pre-trained embedding

class LSTM_2(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_matrix=None,
        embedding_dim=300,
        hidden_dim=256,
        num_labels=3
    ):

        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
                embedding_matrix,
                freeze=False,
                padding_idx=0
            )

        embedding_dim = embedding_matrix.shape[1]

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(
            hidden_dim,
            num_labels
        )

    def forward(self, x):

        x = self.embedding(x)

        x, _ = self.lstm(x)

        x = self.dropout(x)

        x = self.fc(x)

        return x

In [ ]:
# BiLSTM with random embedding

class BiLSTM_1(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=300,
        hidden_dim=256,
        num_labels=3
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(
            hidden_dim * 2,
            num_labels
        )

    def forward(self, x):

        emb = self.embedding(x)

        out, _ = self.lstm(emb)

        out = self.dropout(out)

        return self.fc(out)

In [ ]:
# BiLSTM with pre-trained embedding

class BiLSTM_2(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_matrix,
        hidden_dim=256,
        num_labels=3
    ):

        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=False,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embedding_matrix.shape[1],
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(
            hidden_dim * 2,
            num_labels
        )

    def forward(self, x):

        x = self.embedding(x)

        x, _ = self.lstm(x)

        x = self.dropout(x)

        return self.fc(x)

# Training

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    model_name,
    epochs=30,
    lr=0.0005,
    patience=4
):

    criterion = nn.CrossEntropyLoss(
        ignore_index=-100
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    best_f1 = 0
    patience_counter = 0

    for epoch in range(epochs):

        model.train()

        train_loss = 0

        for batch in train_loader:

            inputs = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            outputs = model(inputs)

            loss = criterion(
                outputs.view(-1, 3),
                labels.view(-1)
            )

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        metrics = evaluate_model(
            model,
            val_loader
        )

        val_f1 = metrics["f1"]

        print(
            f"Epoch {epoch+1} | "
            f"Loss {train_loss/len(train_loader):.4f} | "
            f"Val F1 {val_f1:.4f}"
        )

        if val_f1 > best_f1:

            best_f1 = val_f1

            model_path = MODEL_DIR / f"{model_name}.pt"

            torch.save(
                model.state_dict(),
                model_path
            )

            patience_counter = 0

        else:

            patience_counter += 1

        if patience_counter >= patience:

            print("Early stopping")
            break

    model.load_state_dict(
        torch.load(model_path)
    )

    return model

In [ ]:
def evaluate_model(model, loader):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for batch in loader:

            inputs = batch["input_ids"].to(device)

            labels = batch["labels"]

            mask = batch["mask"]

            outputs = model(inputs)

            preds = outputs.argmax(-1).cpu()

            for pred_seq, label_seq, mask_seq in zip(
                preds,
                labels,
                mask
            ):

                pred_tags = []
                true_tags = []

                for p, l, m in zip(
                    pred_seq,
                    label_seq,
                    mask_seq
                ):

                    if m == 0:
                        continue

                    pred_tags.append(
                        idx2label[int(p)]
                    )

                    true_tags.append(
                        idx2label[int(l)]
                    )

                all_preds.append(pred_tags)
                all_labels.append(true_tags)

    return {
        "precision":
            precision_score(
                all_labels,
                all_preds
            ),
        "recall":
            recall_score(
                all_labels,
                all_preds
            ),
        "f1":
            f1_score(
                all_labels,
                all_preds
            )
    }

## Training - Random embedding

In [ ]:
# RNN

rnn_random = RNN_1(
    vocab_size=len(word2idx)
).to(device)

In [ ]:
print(rnn_random)

RNN_1(
  (embedding): Embedding(3940, 100, padding_idx=0)
  (rnn): RNN(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)


In [ ]:
train_model(
    rnn_random,
    train_loader,
    val_loader,
    model_name="rnn_random"
);

Epoch 1 | Loss 0.6208 | Val F1 0.3189
Epoch 2 | Loss 0.3974 | Val F1 0.4092
Epoch 3 | Loss 0.3452 | Val F1 0.4704
Epoch 4 | Loss 0.3139 | Val F1 0.4876
Epoch 5 | Loss 0.2878 | Val F1 0.5237
Epoch 6 | Loss 0.2621 | Val F1 0.5049
Epoch 7 | Loss 0.2427 | Val F1 0.5192
Epoch 8 | Loss 0.2237 | Val F1 0.5419
Epoch 9 | Loss 0.2031 | Val F1 0.5692
Epoch 10 | Loss 0.1885 | Val F1 0.5848
Epoch 11 | Loss 0.1713 | Val F1 0.5953
Epoch 12 | Loss 0.1567 | Val F1 0.6059
Epoch 13 | Loss 0.1426 | Val F1 0.6012
Epoch 14 | Loss 0.1311 | Val F1 0.6236
Epoch 15 | Loss 0.1198 | Val F1 0.6134
Epoch 16 | Loss 0.1071 | Val F1 0.6279
Epoch 17 | Loss 0.0956 | Val F1 0.6422
Epoch 18 | Loss 0.0860 | Val F1 0.6374
Epoch 19 | Loss 0.0781 | Val F1 0.6405
Epoch 20 | Loss 0.0693 | Val F1 0.6365
Epoch 21 | Loss 0.0629 | Val F1 0.6436
Epoch 22 | Loss 0.0538 | Val F1 0.6518
Epoch 23 | Loss 0.0479 | Val F1 0.6688
Epoch 24 | Loss 0.0418 | Val F1 0.6562
Epoch 25 | Loss 0.0373 | Val F1 0.6551
Epoch 26 | Loss 0.0320 | Val F1 0.

In [ ]:
rnn_results1 = evaluate_model(
    rnn_random,
    test_loader
)
print(rnn_results1)

{'precision': np.float64(0.7296511627906976), 'recall': np.float64(0.6519480519480519), 'f1': np.float64(0.6886145404663924)}


In [ ]:
# LSTM

lstm_random = LSTM_1(
    vocab_size=len(word2idx)
).to(device)

In [ ]:
print(lstm_random)

LSTM_1(
  (embedding): Embedding(3940, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)


In [ ]:
train_model(
    lstm_random,
    train_loader,
    val_loader,
    model_name="lstm_random"
);

Epoch 1 | Loss 0.7262 | Val F1 0.0568
Epoch 2 | Loss 0.4488 | Val F1 0.2228
Epoch 3 | Loss 0.3699 | Val F1 0.4250
Epoch 4 | Loss 0.3171 | Val F1 0.5062
Epoch 5 | Loss 0.2776 | Val F1 0.5373
Epoch 6 | Loss 0.2451 | Val F1 0.5832
Epoch 7 | Loss 0.2194 | Val F1 0.6206
Epoch 8 | Loss 0.1933 | Val F1 0.6169
Epoch 9 | Loss 0.1738 | Val F1 0.6230
Epoch 10 | Loss 0.1540 | Val F1 0.6375
Epoch 11 | Loss 0.1337 | Val F1 0.6633
Epoch 12 | Loss 0.1166 | Val F1 0.6600
Epoch 13 | Loss 0.0984 | Val F1 0.6614
Epoch 14 | Loss 0.0862 | Val F1 0.6559
Epoch 15 | Loss 0.0719 | Val F1 0.6688
Epoch 16 | Loss 0.0607 | Val F1 0.6615
Epoch 17 | Loss 0.0509 | Val F1 0.6656
Epoch 18 | Loss 0.0436 | Val F1 0.6667
Epoch 19 | Loss 0.0365 | Val F1 0.6585
Early stopping


In [ ]:
lstm_results1 = evaluate_model(
    lstm_random,
    test_loader
)
print(lstm_results1)

{'precision': np.float64(0.6898550724637681), 'recall': np.float64(0.6181818181818182), 'f1': np.float64(0.6520547945205479)}


In [ ]:
# BiLSTM

bilstm_random = BiLSTM_1(
    vocab_size=len(word2idx)
).to(device)

In [ ]:
print(bilstm_random)

BiLSTM_1(
  (embedding): Embedding(3940, 300, padding_idx=0)
  (lstm): LSTM(300, 256, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=3, bias=True)
)


In [ ]:
train_model(
    bilstm_random,
    train_loader,
    val_loader,
    model_name="bilstm_random"
);

Epoch 1 | Loss 0.5110 | Val F1 0.3476
Epoch 2 | Loss 0.2814 | Val F1 0.5406
Epoch 3 | Loss 0.2082 | Val F1 0.5983
Epoch 4 | Loss 0.1573 | Val F1 0.6863
Epoch 5 | Loss 0.1170 | Val F1 0.7164
Epoch 6 | Loss 0.0844 | Val F1 0.7143
Epoch 7 | Loss 0.0578 | Val F1 0.7169
Epoch 8 | Loss 0.0409 | Val F1 0.7208
Epoch 9 | Loss 0.0289 | Val F1 0.7201
Epoch 10 | Loss 0.0202 | Val F1 0.7416
Epoch 11 | Loss 0.0152 | Val F1 0.7389
Epoch 12 | Loss 0.0116 | Val F1 0.7496
Epoch 13 | Loss 0.0074 | Val F1 0.7431
Epoch 14 | Loss 0.0053 | Val F1 0.7436
Epoch 15 | Loss 0.0041 | Val F1 0.7432
Epoch 16 | Loss 0.0034 | Val F1 0.7419
Early stopping


In [ ]:
bilstm_results1 = evaluate_model(
    bilstm_random,
    test_loader
)
print(bilstm_results1)

{'precision': np.float64(0.7926829268292683), 'recall': np.float64(0.6753246753246753), 'f1': np.float64(0.7293127629733521)}


# Training - Pre-trained embedding

In [ ]:
# RNN

rnn_pretrained = RNN_2(
    vocab_size=len(word2idx),
    embedding_matrix=embedding_matrix
).to(device)

In [ ]:
print(rnn_pretrained)

RNN_2(
  (embedding): Embedding(3940, 300, padding_idx=0)
  (rnn): RNN(300, 256, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=3, bias=True)
)


In [ ]:
train_model(
    rnn_pretrained,
    train_loader,
    val_loader,
    model_name="rnn_pretrained"
);

Epoch 1 | Loss 0.3540 | Val F1 0.5821
Epoch 2 | Loss 0.1930 | Val F1 0.6499
Epoch 3 | Loss 0.1531 | Val F1 0.7074
Epoch 4 | Loss 0.1263 | Val F1 0.7003
Epoch 5 | Loss 0.1033 | Val F1 0.7175
Epoch 6 | Loss 0.0860 | Val F1 0.7257
Epoch 7 | Loss 0.0702 | Val F1 0.6999
Epoch 8 | Loss 0.0562 | Val F1 0.7324
Epoch 9 | Loss 0.0479 | Val F1 0.6959
Epoch 10 | Loss 0.0362 | Val F1 0.6938
Epoch 11 | Loss 0.0280 | Val F1 0.6785
Epoch 12 | Loss 0.0222 | Val F1 0.6863
Early stopping


In [ ]:
rnn_results2 = evaluate_model(
    rnn_pretrained,
    test_loader
)
print(rnn_results2)

{'precision': np.float64(0.6860759493670886), 'recall': np.float64(0.7038961038961039), 'f1': np.float64(0.6948717948717948)}


In [ ]:
# LSTM

lstm_pretrained = LSTM_2(
    vocab_size=len(word2idx),
    embedding_matrix=embedding_matrix
).to(device)

In [ ]:
print(lstm_pretrained)

LSTM_2(
  (embedding): Embedding(3940, 300, padding_idx=0)
  (lstm): LSTM(300, 256, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=3, bias=True)
)


In [ ]:
train_model(
    lstm_pretrained,
    train_loader,
    val_loader,
    model_name="lstm_pretrained"
);

Epoch 1 | Loss 0.4594 | Val F1 0.4794
Epoch 2 | Loss 0.1726 | Val F1 0.6657
Epoch 3 | Loss 0.1105 | Val F1 0.6909
Epoch 4 | Loss 0.0854 | Val F1 0.7066
Epoch 5 | Loss 0.0694 | Val F1 0.6997
Epoch 6 | Loss 0.0582 | Val F1 0.7006
Epoch 7 | Loss 0.0468 | Val F1 0.7089
Epoch 8 | Loss 0.0378 | Val F1 0.7039
Epoch 9 | Loss 0.0314 | Val F1 0.7036
Epoch 10 | Loss 0.0276 | Val F1 0.6977
Epoch 11 | Loss 0.0211 | Val F1 0.6965
Early stopping


In [ ]:
lstm_results2 = evaluate_model(
    lstm_pretrained,
    test_loader
)
print(lstm_results2)

{'precision': np.float64(0.7150127226463104), 'recall': np.float64(0.7298701298701299), 'f1': np.float64(0.7223650385604113)}


In [ ]:
# BiLSTM

bilstm_pretrained = BiLSTM_2(
    vocab_size=len(word2idx),
    embedding_matrix=embedding_matrix
).to(device)

In [ ]:
print(bilstm_pretrained)

BiLSTM_2(
  (embedding): Embedding(3940, 300, padding_idx=0)
  (lstm): LSTM(300, 256, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=3, bias=True)
)


In [ ]:
train_model(
    bilstm_pretrained,
    train_loader,
    val_loader,
    model_name="bilstm_pretrained"
);

Epoch 1 | Loss 0.3739 | Val F1 0.5543
Epoch 2 | Loss 0.1171 | Val F1 0.7165
Epoch 3 | Loss 0.0735 | Val F1 0.7182
Epoch 4 | Loss 0.0558 | Val F1 0.7340
Epoch 5 | Loss 0.0439 | Val F1 0.7390
Epoch 6 | Loss 0.0362 | Val F1 0.7352
Epoch 7 | Loss 0.0273 | Val F1 0.7500
Epoch 8 | Loss 0.0219 | Val F1 0.7400
Epoch 9 | Loss 0.0170 | Val F1 0.7326
Epoch 10 | Loss 0.0133 | Val F1 0.7390
Epoch 11 | Loss 0.0095 | Val F1 0.7253
Early stopping


In [ ]:
bilstm_results2 = evaluate_model(
    bilstm_pretrained,
    test_loader
)
print(bilstm_results2)

{'precision': np.float64(0.7594594594594595), 'recall': np.float64(0.7298701298701299), 'f1': np.float64(0.7443708609271524)}


# Evaluation

In [ ]:
# Evaluation with random embedding

results_random = pd.DataFrame({
    "Model": [
        "RNN",
        "LSTM",
        "BiLSTM"
    ],
    "Precision": [
        rnn_results1["precision"],
        lstm_results1["precision"],
        bilstm_results1["precision"]
    ],
    "Recall": [
        rnn_results1["recall"],
        lstm_results1["recall"],
        bilstm_results1["recall"]
    ],
    "F1": [
        rnn_results1["f1"],
        lstm_results1["f1"],
        bilstm_results1["f1"]
    ]
})

print(results_random)

    Model  Precision    Recall        F1
0     RNN   0.729651  0.651948  0.688615
1    LSTM   0.689855  0.618182  0.652055
2  BiLSTM   0.792683  0.675325  0.729313


In [ ]:
# Evaluation with pre-trained embedding

results_pretrained = pd.DataFrame({
    "Model": [
        "RNN",
        "LSTM",
        "BiLSTM"
    ],
    "Precision": [
        rnn_results2["precision"],
        lstm_results2["precision"],
        bilstm_results2["precision"]
    ],
    "Recall": [
        rnn_results2["recall"],
        lstm_results2["recall"],
        bilstm_results2["recall"]
    ],
    "F1": [
        rnn_results2["f1"],
        lstm_results2["f1"],
        bilstm_results2["f1"]
    ]
})

print(results_pretrained)

    Model  Precision    Recall        F1
0     RNN   0.686076  0.703896  0.694872
1    LSTM   0.715013  0.729870  0.722365
2  BiLSTM   0.759459  0.729870  0.744371


In [ ]:
# To load models in other notebooks

#model.load_state_dict(torch.load(MODEL_DIR / "rnn_pretrained.pt"))
#model.eval()